In [1]:
import carla
import random 
import cv2
import time 
import numpy as np
import math 

In [2]:
def get_outer_road_lanes(center_waypoint, max_lanes=10):
    """Sweeps to find the absolute leftmost and rightmost drivable lanes."""
    leftmost_lane = center_waypoint
    rightmost_lane = center_waypoint
    
    # Sweep all the way to the Left edge of the road
    left_wp = center_waypoint.get_left_lane()
    lanes_checked = 0
    while left_wp and left_wp.lane_type == carla.LaneType.Driving and lanes_checked < max_lanes:
        leftmost_lane = left_wp
        left_wp = left_wp.get_left_lane()
        lanes_checked += 1
        
    # Sweep all the way to the Right edge of the road
    right_wp = center_waypoint.get_right_lane()
    lanes_checked = 0
    while right_wp and right_wp.lane_type == carla.LaneType.Driving and lanes_checked < max_lanes:
        rightmost_lane = right_wp
        right_wp = right_wp.get_right_lane()
        lanes_checked += 1
        
    return leftmost_lane, rightmost_lane

def get_specific_edge(waypoint, edge_side="left", z_offset=0.5):
    """Calculates ONLY the requested edge (left or right) of a specific lane."""
    center_loc = waypoint.transform.location + carla.Location(z=z_offset)
    half_width = waypoint.lane_width / 2.0
    right_vector = waypoint.transform.get_right_vector()
    
    if edge_side == "left":
        return center_loc - carla.Location(x=right_vector.x * half_width, y=right_vector.y * half_width, z=0.0)
    else: # "right"
        return center_loc + carla.Location(x=right_vector.x * half_width, y=right_vector.y * half_width, z=0.0)

In [3]:
def calculate_wheelbase(vehicle):
    """
    Calculates the physical wheelbase of a CARLA vehicle.
    This only needs to be executed once after the vehicle is spawned.
    
    Args:
        vehicle: carla.Vehicle instance.
        
    Returns:
        float: The wheelbase of the vehicle in meters.
    """
    physics = vehicle.get_physics_control()
    
    # Wheels: 0=Front Left, 1=Front Right, 2=Rear Left, 3=Rear Right
    # Positions are in centimeters, so we divide by 100 to convert to meters
    pos_fl = physics.wheels[0].position / 100.0 
    pos_rl = physics.wheels[2].position / 100.0
    
    # Calculate the 3D Euclidean distance between the front and rear left wheels
    wheelbase = math.sqrt((pos_fl.x - pos_rl.x)**2 + (pos_fl.y - pos_rl.y)**2 + (pos_fl.z - pos_rl.z)**2)
    
    return wheelbase

In [4]:
def predict_full_turn_trajectory(vehicle, wheelbase, turn_direction="left", time_horizon=2.0, dt=0.1):
    """
    Predicts the trajectory of a CARLA vehicle under full steering lock.
    
    Args:
        vehicle: carla.Vehicle instance.
        wheelbase: float, the pre-calculated wheelbase in meters.
        turn_direction: String, "left" or "right".
        time_horizon: How many seconds into the future to predict.
        dt: Time step for the prediction in seconds.
        
    Returns:
        List of carla.Location objects representing the future trajectory points.
    """
    # 1. Gather dynamic vehicle state
    transform = vehicle.get_transform()
    velocity = vehicle.get_velocity()
    physics = vehicle.get_physics_control() 
    
    # Calculate current speed in m/s
    speed = math.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)
    
    # 2. Get Maximum Steering Angle (Delta)
    max_steer_deg = physics.wheels[0].max_steer_angle
    max_steer_rad = math.radians(max_steer_deg)
    
    # Apply direction (CARLA uses left-handed coordinates: positive yaw is right)
    if turn_direction.lower() == "left":
        steer_angle = -max_steer_rad
    else: 
        steer_angle = max_steer_rad

    # 3. Initialize Starting Variables
    x = transform.location.x
    y = transform.location.y
    z = transform.location.z 
    yaw = math.radians(transform.rotation.yaw)
    
    trajectory = []
    
    # 4. Step through time to generate the predicted path
    steps = int(time_horizon / dt)
    for _ in range(steps):
        trajectory.append(carla.Location(x=x, y=y, z=z))
        
        # Calculate yaw rate using the pre-calculated wheelbase
        if wheelbase > 0:
            yaw_rate = (speed / wheelbase) * math.tan(steer_angle)
        else:
            yaw_rate = 0
            
        # Update coordinates using Euler integration
        x += speed * math.cos(yaw) * dt
        y += speed * math.sin(yaw) * dt
        yaw += yaw_rate * dt
        
    return trajectory

In [5]:
def predict_90deg_trajectory(vehicle, wheelbase, turn_direction="right", steer_intensity=1.0, time_horizon=4.0, dt=0.1):
    """
    Predicts a turn that stops steering after exactly 90 degrees.
    
    Args:
        vehicle: carla.Vehicle instance.
        wheelbase: float, pre-calculated wheelbase in meters.
        turn_direction: String, "left" or "right".
        steer_intensity: Float from 0.0 (straight) to 1.0 (full lock).
        time_horizon: Seconds into the future to predict.
        dt: Time step.
    """
    transform = vehicle.get_transform()
    velocity = vehicle.get_velocity()
    physics = vehicle.get_physics_control() 
    
    speed = math.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)
    
    # Avoid errors if the car is completely stopped
    if speed < 0.1:
        return []

    # 1. Clamp intensity to ensure it stays between 0% and 100%
    steer_intensity = max(0.0, min(1.0, steer_intensity))

    # 2. Apply the intensity to the vehicle's maximum physical steering limit
    max_steer_rad = math.radians(physics.wheels[0].max_steer_angle)
    applied_steer_rad = max_steer_rad * steer_intensity
    
    # Apply direction
    if turn_direction.lower() == "left":
        steer_angle = -applied_steer_rad
    else: 
        steer_angle = applied_steer_rad

    x = transform.location.x
    y = transform.location.y
    z = transform.location.z 
    current_yaw = math.radians(transform.rotation.yaw)
    
    # Calculate the fixed turning rate using the dampened steering angle
    yaw_rate = (speed / wheelbase) * math.tan(steer_angle) if wheelbase > 0 else 0
    
    trajectory = []
    accumulated_yaw = 0.0
    target_yaw_change = math.pi / 2.0  # 90 degrees in radians
    
    steps = int(time_horizon / dt)
    
    for _ in range(steps):
        trajectory.append(carla.Location(x=x, y=y, z=z))
        
        # If we haven't turned 90 degrees yet, keep turning
        if abs(accumulated_yaw) < target_yaw_change:
            active_yaw_rate = yaw_rate
        else:
            # We hit 90 degrees. Center the steering wheel (go straight).
            active_yaw_rate = 0.0 
            
        x += speed * math.cos(current_yaw) * dt
        y += speed * math.sin(current_yaw) * dt
        
        current_yaw += active_yaw_rate * dt
        accumulated_yaw += active_yaw_rate * dt
        
    return trajectory

In [6]:
def predict_trajectory_to_impact(vehicle, wheelbase, carla_map, turn_direction="right", steer_intensity=1.0, time_horizon=4.0, dt=0.1):
    """
    Predicts a trajectory and stops exactly when it hits a road boundary.
    Returns the trajectory points and the distance traveled to impact.
    """
    transform = vehicle.get_transform()
    velocity = vehicle.get_velocity()
    physics = vehicle.get_physics_control() 
    
    speed = math.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)
    
    # Return empty if stationary
    if speed < 0.1:
        return [], 0.0

    steer_intensity = max(0.0, min(1.0, steer_intensity))
    max_steer_rad = math.radians(physics.wheels[0].max_steer_angle)
    applied_steer_rad = max_steer_rad * steer_intensity
    
    if turn_direction.lower() == "left":
        steer_angle = -applied_steer_rad
    else: 
        steer_angle = applied_steer_rad

    x = transform.location.x
    y = transform.location.y
    z = transform.location.z 
    current_yaw = math.radians(transform.rotation.yaw)
    
    yaw_rate = (speed / wheelbase) * math.tan(steer_angle) if wheelbase > 0 else 0
    
    # Initialize variables
    trajectory = []
    accumulated_yaw = 0.0
    target_yaw_change = math.pi / 2.0  # 90 degrees max turn
    distance_traveled = 0.0
    
    steps = int(time_horizon / dt)
    
    for _ in range(steps):
        # 1. Update kinematics
        if abs(accumulated_yaw) < target_yaw_change:
            active_yaw_rate = yaw_rate
        else:
            active_yaw_rate = 0.0 
            
        x += speed * math.cos(current_yaw) * dt
        y += speed * math.sin(current_yaw) * dt
        
        current_yaw += active_yaw_rate * dt
        accumulated_yaw += active_yaw_rate * dt
        
        # Track arc length
        distance_traveled += speed * dt 
        
        current_loc = carla.Location(x=x, y=y, z=z)
        trajectory.append(current_loc)
        
        # 2. Collision / Boundary Check
        # project_to_road=False ensures it returns None if the point is off the lane
        wp = carla_map.get_waypoint(current_loc, project_to_road=False, lane_type=carla.LaneType.Driving)
        
        # If the waypoint is None, the point has crossed out of the drivable road boundaries
        if wp is None:
            break # Stop the simulation instantly
            
    return trajectory, distance_traveled

In [7]:
# Connect to the client
client = carla.Client('localhost', 2000)
client.set_timeout(10.0)
client.load_world('Town04')
world = client.get_world()
carla_map = world.get_map()

print("Successfully connected to CARLA and loaded the map!")

Successfully connected to CARLA and loaded the map!


In [11]:
# Get blueprints and spawn points
blueprint_library = world.get_blueprint_library()
vehicle_bp = random.choice(blueprint_library.filter('vehicle.*'))
spawn_point = random.choice(carla_map.get_spawn_points())

# Spawn the vehicle and set it to drive automatically
vehicle = world.spawn_actor(vehicle_bp, spawn_point)
vehicle.set_autopilot(True)

print(f"Spawned {vehicle.type_id} at {spawn_point.location}")

Spawned vehicle.tesla.model3 at Location(x=-391.838043, y=12.740335, z=0.281942)


In [9]:
spectator = world.get_spectator() 
ego_wheelbase = calculate_wheelbase(vehicle)
print("Tracking vehicle and drawing absolute road boundaries. Click 'Stop' to end.")

last_draw_time = 0
draw_interval = 0.1  

try:
    while True:
        world.wait_for_tick() 

        # 1. --- SPECTATOR TRACKING ---
        transform = vehicle.get_transform()
        yaw = math.radians(transform.rotation.yaw)
        x_offset = -6.0 * math.cos(yaw)
        y_offset = -6.0 * math.sin(yaw)
        
        camera_location = transform.location + carla.Location(x=x_offset, y=y_offset, z=2.5)
        camera_rotation = carla.Rotation(pitch=-15.0, yaw=transform.rotation.yaw, roll=0.0)
        spectator.set_transform(carla.Transform(camera_location, camera_rotation))


        # 2. --- TRAJECTORY PREDICTION & DRAWING ---
        current_time = time.time()
        if current_time - last_draw_time >= draw_interval:
            last_draw_time = current_time 

            left_trajectory, l_CL = predict_trajectory_to_impact(vehicle, ego_wheelbase, carla_map, "left", steer_intensity=0.3)
            right_trajectory, l_CR = predict_trajectory_to_impact(vehicle, ego_wheelbase, carla_map, "right", steer_intensity=0.3)
            
            if (l_CL + l_CR) > 0:
                slider_value = l_CL / (l_CL + l_CR)
            else:
                slider_value = 0.5 # Default center

            # Ensure the value stays bounded between 0 and 1
            slider_value = max(0.0, min(1.0, slider_value))
            display_text = f"L: {l_CL:.1f}m | R: {l_CR:.1f}m | Slider: {slider_value:.2f}"

            text_location = vehicle.get_transform().location + carla.Location(z=2.0)

            # Draw the string in the simulation
            world.debug.draw_string(
                text_location, 
                display_text, 
                draw_shadow=True, 
                color=carla.Color(255, 255, 255), # White text
                life_time=0.15 # Set this slightly higher than your loop dt to prevent flickering
            )
            # Draw Left Turn (Blue)
            for loc in left_trajectory:
                draw_loc = loc + carla.Location(z=0.5) # Lift slightly off the ground
                world.debug.draw_point(draw_loc, size=0.08, color=carla.Color(0, 0, 255), life_time=0.15)
                
            # Draw Right Turn (Yellow)
            for loc in right_trajectory:
                draw_loc = loc + carla.Location(z=0.5)
                world.debug.draw_point(draw_loc, size=0.08, color=carla.Color(255, 255, 0), life_time=0.15)

except KeyboardInterrupt:
    print("\nVisualization stopped by user.")
finally:
    if 'vehicle' in locals() and vehicle is not None:
        vehicle.destroy()
        print("Vehicle destroyed. Clean exit.")

Tracking vehicle and drawing absolute road boundaries. Click 'Stop' to end.

Visualization stopped by user.
Vehicle destroyed. Clean exit.


In [ ]:
lookahead_distance = 2.0   # Distance between waypoints
num_points = 15            # Number of waypoints to look ahead for drawing boundaries
# You can lookahead 30m
spectator = world.get_spectator() 
ego_wheelbase = calculate_wheelbase(vehicle)
print("Tracking vehicle and drawing absolute road boundaries. Click 'Stop' to end.")

last_draw_time = 0
draw_interval = 0.1  

try:
    while True:
        world.wait_for_tick() 

        # 1. --- SPECTATOR TRACKING ---
        transform = vehicle.get_transform()
        yaw = math.radians(transform.rotation.yaw)
        x_offset = -6.0 * math.cos(yaw)
        y_offset = -6.0 * math.sin(yaw)
        
        camera_location = transform.location + carla.Location(x=x_offset, y=y_offset, z=2.5)
        camera_rotation = carla.Rotation(pitch=-15.0, yaw=transform.rotation.yaw, roll=0.0)
        spectator.set_transform(carla.Transform(camera_location, camera_rotation))


        # 2. --- ROAD BOUNDARY DRAWING ---
        current_time = time.time()
        if current_time - last_draw_time >= draw_interval:
            last_draw_time = current_time 
            
            current_waypoint = carla_map.get_waypoint(transform.location)
            next_waypoint = current_waypoint

            for i in range(num_points):
                next_waypoints_list = next_waypoint.next(lookahead_distance)
                if not next_waypoints_list:
                    break 
                
                next_waypoint = next_waypoints_list[0]
                
                # 1. Find the outermost lanes
                leftmost_wp, rightmost_wp = get_outer_road_lanes(next_waypoint)

                # 2. Calculate the center of the car's path, and the absolute road edges
                ego_center = next_waypoint.transform.location + carla.Location(z=0.5)
                road_left_edge = get_specific_edge(leftmost_wp, edge_side="left")
                road_right_edge = get_specific_edge(rightmost_wp, edge_side="right")

                # 3. Draw them
                world.debug.draw_point(ego_center, size=0.08, color=carla.Color(255, 0, 0), life_time=0.15)
                world.debug.draw_point(road_left_edge, size=0.08, color=carla.Color(0, 255, 0), life_time=0.15)
                world.debug.draw_point(road_right_edge, size=0.08, color=carla.Color(0, 255, 0), life_time=0.15)

                # 2. PREDICT & DRAW FULL-LOCK TRAJECTORIES
            # Predicting 2 seconds into the future
            #left_trajectory = predict_90deg_trajectory(vehicle, ego_wheelbase, "left", 0.3)
            #right_trajectory = predict_90deg_trajectory(vehicle, ego_wheelbase, "right", 0.3)
            

            left_trajectory, l_CL = predict_trajectory_to_impact(vehicle, ego_wheelbase, carla_map, "left", steer_intensity=0.3)
            right_trajectory, l_CR = predict_trajectory_to_impact(vehicle, ego_wheelbase, carla_map, "right", steer_intensity=0.3)
            # Draw Left Turn (Blue)
            for loc in left_trajectory:
                draw_loc = loc + carla.Location(z=0.5) # Lift slightly off the ground
                world.debug.draw_point(draw_loc, size=0.08, color=carla.Color(0, 0, 255), life_time=0.15)
                
            # Draw Right Turn (Yellow)
            for loc in right_trajectory:
                draw_loc = loc + carla.Location(z=0.5)
                world.debug.draw_point(draw_loc, size=0.08, color=carla.Color(255, 255, 0), life_time=0.15)

except KeyboardInterrupt:
    print("\nVisualization stopped by user.")
finally:
    if 'vehicle' in locals() and vehicle is not None:
        vehicle.destroy()
        print("Vehicle destroyed. Clean exit.")

Tracking vehicle and drawing absolute road boundaries. Click 'Stop' to end.

Visualization stopped by user.
Vehicle destroyed. Clean exit.


In [ ]:
import pygame

pygame.init()
display = pygame.display.set_mode((300, 200))
pygame.display.set_caption("CLICK HERE TO DRIVE")

print("IMPORTANT: Click the new 'CLICK HERE TO DRIVE' window that just opened!")
print("Controls: W (Gas), S (Brake), A/D (Steer), Q (Hold for Reverse)")

# Assuming `world` and `vehicle` are already defined in your environment
carla_map = world.get_map()
vehicle.set_autopilot(False)
control = carla.VehicleControl()
spectator = world.get_spectator() 

ego_wheelbase = calculate_wheelbase(vehicle)
steer_intensity_setting = 0.4 # Adjust this to make the predicted turn sharper or wider

last_draw_time = 0
draw_interval = 0.1  

try:
    while True:
        world.wait_for_tick() 

        # --- KEYBOARD CONTROL LOGIC ---
        pygame.event.pump() 
        keys = pygame.key.get_pressed()

        # Throttle / Brake / Steer / Reverse
        control.throttle = min(control.throttle + 0.05, 1.0) if keys[pygame.K_w] else 0.0
        control.brake = min(control.brake + 0.2, 1.0) if keys[pygame.K_s] else 0.0
        
        if keys[pygame.K_a]:
            control.steer = max(control.steer - 0.05, -1.0)
        elif keys[pygame.K_d]:
            control.steer = min(control.steer + 0.05, 1.0)
        else:
            control.steer = 0.0 
            
        control.reverse = keys[pygame.K_q]
        vehicle.apply_control(control)

        # Pygame exit check
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                raise KeyboardInterrupt

        # --- SPECTATOR TRACKING ---
        transform = vehicle.get_transform()
        yaw = math.radians(transform.rotation.yaw)
        x_offset = -6.0 * math.cos(yaw)
        y_offset = -6.0 * math.sin(yaw)
        
        camera_location = transform.location + carla.Location(x=x_offset, y=y_offset, z=2.5)
        camera_rotation = carla.Rotation(pitch=-15.0, yaw=transform.rotation.yaw, roll=0.0)
        spectator.set_transform(carla.Transform(camera_location, camera_rotation))

        # --- RAD SLIDER & TRAJECTORY DRAWING ---
        current_time = time.time()
        if current_time - last_draw_time >= draw_interval:
            last_draw_time = current_time 
            
            # 1. Calculate Trajectories to Impact
            left_traj, l_CL = predict_trajectory_to_impact(vehicle, ego_wheelbase, carla_map, "left", steer_intensity=steer_intensity_setting)
            right_traj, l_CR = predict_trajectory_to_impact(vehicle, ego_wheelbase, carla_map, "right", steer_intensity=steer_intensity_setting)

            # 2. Calculate Slider Value
            if (l_CL + l_CR) > 0:
                slider_value = l_CL / (l_CL + l_CR)
            else:
                # Edge case: Car is stopped OR already completely off the road
                current_wp = carla_map.get_waypoint(transform.location, project_to_road=False, lane_type=carla.LaneType.Driving)
                if current_wp is None:
                    # Car is off track. Default to 0.5 or implement specific left/right detection here.
                    slider_value = 0.5 
                else:
                    slider_value = 0.5 # Car is just stopped on the road
            
            slider_value = max(0.0, min(1.0, slider_value))

            # 3. Draw 3D Text HUD (Floating above the car)
            display_text = f"L: {l_CL:.1f}m | R: {l_CR:.1f}m | RAD Slider: {slider_value:.2f}"
            text_location = transform.location + carla.Location(z=2.0)
            
            world.debug.draw_string(
                text_location, 
                display_text, 
                draw_shadow=True, 
                color=carla.Color(255, 255, 255), 
                life_time=0.15
            )

            # 4. Draw Trajectory Dots
            for loc in left_traj:
                world.debug.draw_point(loc + carla.Location(z=0.5), size=0.08, color=carla.Color(0, 0, 255), life_time=0.15)
                
            for loc in right_traj:
                world.debug.draw_point(loc + carla.Location(z=0.5), size=0.08, color=carla.Color(255, 255, 0), life_time=0.15)

except KeyboardInterrupt:
    print("\nScript stopped by user.")
finally:
    if 'vehicle' in locals() and vehicle is not None:
        vehicle.destroy()
    pygame.quit()
    print("Clean exit.")

pygame 2.6.1 (SDL 2.28.4, Python 3.12.0)
Hello from the pygame community. https://www.pygame.org/contribute.html
IMPORTANT: Click the new 'CLICK HERE TO DRIVE' window that just opened!
Controls: W (Gas), S (Brake), A/D (Steer), Q (Hold for Reverse)

Script stopped by user.
Clean exit.


: 